<a href="https://colab.research.google.com/github/LCaravaggio/NLP/blob/main/notebooks/09_Topic_Modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Activar GPU en colab con "Change runtime type".

# Datos

Vamos a usar el conocido data set 20 Newsgroup. Se trata de un conjunto de datos clásico y muy utilizado en el campo del procesamiento de lenguaje natural (NLP) y el aprendizaje automático. Fue recopilado originalmente por Ken Lang en 1995 y contiene aproximadamente 20.000 mensajes tomados de 20 grupos de noticias (newsgroups) diferentes de Usenet, una red de foros de discusión en línea muy popular antes del auge de la web moderna. Viene cargado en Sklearn.

In [ ]:
from sklearn.datasets import fetch_20newsgroups
docs = fetch_20newsgroups(subset='all',  remove=('headers', 'footers', 'quotes'))['data']

In [ ]:
# Algunos ejemplos:
print(*docs[198:202], sep="\n" + "#"*80 + "\n")

# Bertopic

Vamos a tener que instalar bertopic porque no viene preinstalado en Colab

In [ ]:
!pip install -qU bertopic watermark

In [ ]:
from watermark import watermark
print(watermark())
print(watermark(packages="bertopic,numpy"))

Directamente entrenamos el modelo a ver qué nos da.

In [ ]:
from bertopic import BERTopic

topic_model = BERTopic(language="english", calculate_probabilities=True, verbose=True)
topics, probs = topic_model.fit_transform(docs)

In [ ]:
freq = topic_model.get_topic_info();
freq.head(5)

**PREGUNTA 1**: ¿es conveniente usar GPU en este caso? ¿Por qué?



La salida del modelo no asigna una etiqueta a cada tópico pero nos muestra las palabras más relevantes. En base a esa información uno podría asignarle nombres a los tópicos que le interese. El tópico -1 es el de los outliers, sería un tópico no relevante.

Atributos
Hay varios atributos a los que podés acceder después de haber entrenado tu modelo BERTopic:

|Atributo	 | Descripción |
|------------------------|---------------------------------------------------------------------------------------------|
|topics_	|Los temas que se generan para cada documento después de entrenar o actualizar el modelo de temas.
|probabilities_	|Las probabilidades que se generan para cada documento si se utiliza HDBSCAN.
|topic_sizes_	|El tamaño de cada tema.
|topic_mapper_ |	Una clase para rastrear los temas y sus asignaciones cada vez que se combinan o reducen.
|topic_representations_	|Los n términos principales por tema y sus respectivos valores de c-TF-IDF.
|c_tf_idf_|	La matriz de términos por tema calculada mediante c-TF-IDF.
|topic_labels_|	Las etiquetas predeterminadas para cada tema.
|custom_labels_	|Etiquetas personalizadas para cada tema, generadas mediante .set_topic_labels.
|topic_embeddings_	|Los embeddings de cada tema si se utilizó embedding_model.
|representative_docs_	|Los documentos representativos de cada tema si se utilizó HDBSCAN.

Por ejemplo, para acceder a los temas predichos de los primeros 10 documentos, simplemente ejecutamos lo siguiente:

In [ ]:
topic_model.topics_[:10]

**PREGUNTA 2** ¿qué es un outlier precisamente en el contexto de BERTopic?

## Visualización

In [ ]:
topic_model.visualize_topics()

**PREGUNTA 3** ¿qué representa el gráfico anterior?

La variable probabilities, que se devuelve al ejecutar transform() o fit_transform(), se puede usar para entender con qué grado de confianza BERTopic asigna uno o más tópicos a un documento.

Es decir, no solo te dice qué tópico es el más probable, sino qué tan probable es cada tópico dentro de ese documento. Esta información puede ser útil si un documento está relacionado con varios tópicos o si hay baja certeza sobre la clasificación.

Para visualizar cómo se distribuyen esas probabilidades (es decir, qué tópicos aparecen y con qué peso en un documento específico), simplemente ejecutás:

In [ ]:
topic_model.visualize_distribution(probs[200], min_probability=0.015)

In [ ]:
print(docs[200])

probs[200] indica que estás visualizando la distribución de temas del documento número 200.

min_probability=0.015 filtra los temas que tienen menos de 1.5% de probabilidad para que el gráfico sea más claro (oculta temas irrelevantes o con peso muy bajo).

Esto genera un gráfico interactivo (usualmente un gráfico de barras) que muestra qué temas aparecen en ese documento y con qué fuerza.

## Jerarquía de Tópicos

Los tópicos que fueron generados por BERTopic pueden reducirse jerárquicamente, es decir, pueden agruparse en tópicos más generales.

Para entender mejor la posible estructura jerárquica de los tópicos (por ejemplo, cómo unos tópicos están relacionados o se pueden combinar), se puede usar la función scipy.cluster.hierarchy para:

* Agrupar los tópicos en clústeres jerárquicos.
* Visualizar cómo se conectan o agrupan esos tópicos entre sí.

Esto puede ayudarte a decidir cuántos tópicos finales (nr_topics) querés mantener cuando reduzcas el número total de tópicos generados automáticamente por el modelo.

Por ejemplo, si BERTopic detectó inicialmente 50 tópicos pero muchos son muy parecidos, esta visualización puede ayudarte a ver que tal vez con 20 o 25 tópicos sería suficiente.

In [ ]:
topic_model.visualize_hierarchy(top_n_topics=50)

## Visualización de términos

Podemos visualizar los términos seleccionados de algunos tópicos creando gráficos de barras a partir de los puntajes de c-TF-IDF de cada representación de tópico.

El c-TF-IDF (class-based TF-IDF) es una versión adaptada del TF-IDF tradicional, que mide la importancia de cada palabra dentro de un tópico (en lugar de dentro de un documento).

Así, los términos con mayor c-TF-IDF son los más representativos de ese tópico.

Esto permite:

* Obtener insights (ideas clave) observando los puntajes relativos de c-TF-IDF dentro de un tópico (qué términos lo definen más claramente) y entre tópico (qué los diferencia).
* Comparar de forma sencilla las representaciones de distintos tópico entre sí, viendo qué palabras los definen y en qué grado.

In [ ]:
topic_model.visualize_barchart(top_n_topics=5)

## Similitud de tópicos

Una vez que se han generado los embeddings de tópicos (representaciones numéricas de cada tema), ya sea mediante c-TF-IDF o mediante un modelo de embeddings, se puede construir una matriz de similitud.

Esto produce una matriz que indica qué tan similares son los distintos temas entre sí.

Cada celda de la matriz muestra el grado de similitud entre dos temas.

Esta matriz es útil, por ejemplo, para identificar temas redundantes o agrupar temas relacionados.

In [ ]:
topic_model.visualize_heatmap(n_clusters=20, width=1000, height=1000)

**PREGUNTA 4**: ¿Qué medida de similitud usa la matriz anterior?

## Personalización

BERTopic se caracteriza por su modularidad: podés combinar diferentes técnicas de vectorización, embeddings, reducción de dimensionalidad, clustering, e incluso asignación de tópicos. Al cambiar estos módulos (y no solo sus hiperparámetros), podés obtener resultados más adecuados a la naturaleza de tus textos y al tipo de análisis que querés hacer.

Se pueden personalizar:

1. Vectorización de texto: Convierte los documentos en matrices de términos. Algunas alternativas:
* CountVectorizer (por defecto)
* TfidfVectorizer (mejor si hay mucho ruido o vocabulario disperso)
* Vectorizadores con n-gramas, eliminación de stopwords personalizados o reducción de vocabulario (max_df, min_df)
* Incluso se puede usar un vectorizador preprocesado externo si tenés un pipeline específico

2. Modelo de embeddings: Convierte el texto en vectores semánticos. Algunas opciones:

* all-MiniLM-L6-v2 (rápido y balanceado, por defecto)
* all-mpnet-base-v2 (más preciso, más pesado)
* distiluse-base-multilingual-cased (para textos en múltiples idiomas)
* Modelos propios
* Alternativas no basadas en BERT, como GloVe o spaCy (menos comunes)

3. Reducción de dimensionalidad: Reduce la dimensionalidad de los embeddings para facilitar el clustering:
* UMAP (por defecto, no lineal, configurable con n_neighbors, min_dist, etc.)
* PCA (lineal, útil cuando los embeddings ya son de baja dimensión)
* TSNE (más costoso pero visualmente informativo)
* También se puede omitir esta etapa

4. Modelo de clustering: agrupa documentos similares según sus vectores reducidos:
* HDBSCAN (por defecto, no requiere predefinir el número de clusters, tolerante al ruido)
* KMeans (requiere n_clusters, útil para comparación directa)
* AgglomerativeClustering (jerárquico, útil si esperás relaciones entre grupos)
* SpectralClustering, DBSCAN o modelos propios de clustering también se pueden usar

5. Asignación y representación de tópicos. BERTopic también permite:
* Cambiar el número de tópicos finales (nr_topics)
* Fusionar tópicos jerárquicamente
* Cambiar la representación textual de los tópicos (set_topic_labels, set_topic_representations)

**PREGUNTA 5** ¿Para qué se usa el primer paso de vectorización de texto?

In [ ]:
### Por ejemplo:

# from bertopic import BERTopic
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.decomposition import PCA
# from sklearn.cluster import KMeans
# from sentence_transformers import SentenceTransformer

# # 1. Vectorizador alternativo: TF-IDF en lugar de CountVectorizer
# vectorizer_model = TfidfVectorizer(ngram_range=(1, 3), stop_words="english", max_df=0.95, min_df=5)

# # 2. Embedding model más potente
# embedding_model = SentenceTransformer("all-mpnet-base-v2")  # más preciso pero más pesado

# # 3. Reducción de dimensionalidad con PCA en lugar de UMAP
# from sklearn.decomposition import PCA
# dim_reduction_model = PCA(n_components=15)

# # 4. Clustering con KMeans en lugar de HDBSCAN
# from sklearn.cluster import KMeans
# clustering_model = KMeans(n_clusters=20, random_state=42)

# # 5. Crear el modelo BERTopic con estos componentes
# topic_model = BERTopic(
#     embedding_model=embedding_model,
#     vectorizer_model=vectorizer_model,
#     umap_model=dim_reduction_model,
#     hdbscan_model=clustering_model,
#     language="english",
#     calculate_probabilities=True,
#     verbose=True
# )

# # 6. Entrenamiento del modelo
# topics, probs = topic_model.fit_transform(docs)

# LDA

Preprocesar y vectorizar los documentos para LDA en sklearn requiere una matriz documento-palabra (podemos usar para esto CountVectorizer o TF-IDF).

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Vectorización (podés ajustar stop_words, ngram_range, etc.)
vectorizer = CountVectorizer(stop_words='english')
doc_term_matrix = vectorizer.fit_transform(docs)

Después simplemente podemos ya entrenar el modelo

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

# Definir número de tópicos
n_topics = 10

lda_model = LatentDirichletAllocation(n_components=n_topics,
                                       max_iter=10,
                                       learning_method='online',
                                       random_state=42)
lda_model.fit(doc_term_matrix)

In [ ]:
# Obtener las palabras del vocabulario
terms = vectorizer.get_feature_names_out()

# Mostrar los 10 términos más importantes por tópico
for idx, topic in enumerate(lda_model.components_):
    print(f"\nTópico {idx}:")
    top_indices = topic.argsort()[-10:][::-1]
    top_terms = [terms[i] for i in top_indices]
    print(", ".join(top_terms))

In [ ]:
# Distribución de tópicos por documento
doc_topic_dist = lda_model.transform(doc_term_matrix)

# Por ejemplo: tópico más probable para el primer documento
import numpy as np
print("Tópico dominante en el primer documento:", np.argmax(doc_topic_dist[0]))

**PREGUNTA 6**: ¿qué diferencia conceptual fundamental hay entre LDA y BERTopic?